# 第17章 机器学习项目流程

本 notebook 用一个极小的恒星分类任务说明训练集、测试集、baseline 和模型评估的最小流程。

## 学习目标

- 理解特征、标签、训练集和测试集的基本含义。
- 从一个规则 baseline 过渡到一个简单的数据驱动分类器。
- 认识到小样本结果只适合教学，不适合真实科学结论。

In [ ]:
import csv
import math
from pathlib import Path

data_path = Path("../../data/small/stellar_photometry_demo.csv")
with data_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for row in rows:
    row["bp_rp"] = float(row["bp_rp"])
    row["parallax_mas"] = float(row["parallax_mas"])
    row["g_mag"] = float(row["g_mag"])
    row["absolute_g_mag"] = row["g_mag"] - 10.0 + 5.0 * math.log10(row["parallax_mas"])

train_rows = rows[:6]
test_rows = rows[6:]
print("train:", len(train_rows), "test:", len(test_rows))


## 一个最小的最近中心分类器

下面的实现不依赖任何第三方机器学习库。它只是为了把“特征 -> 训练 -> 预测 -> 评估”这个流程完整跑通。

In [ ]:
def feature_vector(row):
    return (row["bp_rp"], row["absolute_g_mag"])


def build_centroids(training_rows):
    grouped = {}
    for row in training_rows:
        grouped.setdefault(row["stellar_class"], []).append(feature_vector(row))

    centroids = {}
    for label, values in grouped.items():
        x_mean = sum(v[0] for v in values) / len(values)
        y_mean = sum(v[1] for v in values) / len(values)
        centroids[label] = (x_mean, y_mean)
    return centroids


def classify_nearest_centroid(row, centroids):
    x_value, y_value = feature_vector(row)
    best_label = None
    best_distance = None
    for label, (cx, cy) in centroids.items():
        distance = ((x_value - cx) ** 2 + (y_value - cy) ** 2) ** 0.5
        if best_distance is None or distance < best_distance:
            best_distance = distance
            best_label = label
    return best_label


centroids = build_centroids(train_rows)
centroids


In [ ]:
test_predictions = []
for row in test_rows:
    predicted = classify_nearest_centroid(row, centroids)
    test_predictions.append((row["label"], row["stellar_class"], predicted))

test_predictions


In [ ]:
test_accuracy = sum(1 for _, truth, pred in test_predictions if truth == pred) / len(test_predictions)
print("nearest-centroid test accuracy =", round(test_accuracy, 3))


## 如果安装了 scikit-learn

下面这个单元是可选的。如果环境里装了 `scikit-learn`，你可以对比一个标准库实现；如果没有安装，也不会影响整个 notebook 的教学流程。

In [ ]:
try:
    from sklearn.neighbors import KNeighborsClassifier
except ModuleNotFoundError:
    print("scikit-learn 未安装；已跳过 KNN 示例。")
else:
    x_train = [feature_vector(row) for row in train_rows]
    y_train = [row["stellar_class"] for row in train_rows]
    x_test = [feature_vector(row) for row in test_rows]
    y_test = [row["stellar_class"] for row in test_rows]

    model = KNeighborsClassifier(n_neighbors=1)
    model.fit(x_train, y_train)
    print("knn score =", round(model.score(x_test, y_test), 3))


## 小结

这个例子非常小，但已经包含了机器学习工作流的基本骨架：定义特征、划分数据、建立 baseline、训练模型、评估结果。后续章节会把这个流程逐步扩展到更真实的数据规模与更严格的评估。